<a href="https://colab.research.google.com/github/iqra-amer/Flyrank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 3 – Data Contract

**Lane:** Content Refresh Prioritization

This notebook defines the data contract for my lane, validates the selected data slice, builds an initial feature frame, and demonstrates data leakage.

In [ ]:
# Iport the libraries

import os
import duckdb
import pandas as pd
from google.colab import userdata

In [ ]:
# Connect to the hugging face

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "dim_clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "dim_content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    "fact_query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')"
}

print("Connection Successful!")

Connection Successful!


## Data Contract

### 1. What does one row represent?

One row represents the daily performance of a single content page for one client on one reporting date.

### 2. Which table(s) are used?

The primary table is:

- fact_content_daily_performance

### 3. What time window is used?

March 2026.

A middle month was selected because it has historical data before it and future data after it, making it suitable for later modeling.

### 4. What will be predicted?

Whether a content page should be prioritized for refreshing.

For this assignment, a simple proxy label is used.

### 5. What is excluded?

The final month of the panel (June 2026) is excluded because it should remain an unseen test period.

In [ ]:
# Verification Query 1

con.sql(f"""
SELECT
report_date,
client_hash_id,
content_hash_id,
COUNT(*) AS duplicate_count

FROM {TABLES["fact_daily"]}

WHERE month='2026-03'

GROUP BY
report_date,
client_hash_id,
content_hash_id

HAVING COUNT(*)>1
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_count


### Verification 1

No duplicate rows were found.

This confirms that the grain is one content page for one client on one reporting day.

In [ ]:
# Verification Query 2

con.sql(f"""
SELECT

COUNT(*) AS total_rows,
MIN(report_date) AS start_date,
MAX(report_date) AS end_date

FROM {TABLES["fact_daily"]}

WHERE month='2026-03'
""").df()

,total_rows,start_date,end_date
0,9841378,2026-03-01,2026-03-31


### Verification 2

The March 2026 slice contains 9,841,378 daily observations.

The data ranges from 2026-03-01 through 2026-03-31, confirming the complete month is available.

In [ ]:
# Verification Query 3

con.sql(f"""
SELECT

COUNT(*) AS available_rows

FROM {TABLES["fact_daily"]}

WHERE month='2026-03'
AND gsc_data_available IS TRUE
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,available_rows
0,3611061


### Verification 3

3,611,061 rows contain valid Google Search Console data.

Only these rows are used when engineering search-performance features.

In [ ]:
# Feature Engineering

feature_frame = con.sql(f"""
SELECT

client_hash_id,
content_hash_id,

SUM(gsc_impressions) AS total_impressions,
SUM(gsc_clicks) AS total_clicks,
AVG(gsc_sum_position) AS avg_position,
SUM(ga4_sessions) AS total_sessions,
SUM(scroll_events) AS total_scroll_events

FROM {TABLES["fact_daily"]}

WHERE month='2026-03'
AND gsc_data_available IS TRUE

GROUP BY
client_hash_id,
content_hash_id
""").df()

feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,total_impressions,total_clicks,avg_position,total_sessions,total_scroll_events
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,6523.0,7.0,1450.483871,1.0,0.0
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,453.0,0.0,46.967742,0.0,0.0
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,5630.0,6.0,1186.903226,3.0,0.0
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,4944.0,13.0,1185.870968,2.0,0.0
4,client_73cda7b4e4f265ea,content_1855a661b4d36130,429.0,1.0,53.580645,2.0,1.0


## Feature Frame

Each row now represents one content page for one client summarized across March 2026.

### Features

**total_impressions**

Knowable at the decision moment because impressions are historical observations.

**total_clicks**

Knowable because clicks occurred before the refresh decision.

**avg_position**

Knowable because historical ranking is already available.

**total_sessions**

Knowable because website traffic has already occurred.

**total_scroll_events**

Knowable because user engagement is measured before making the refresh decision.

In [ ]:
# Leakage Demostration

# Creating the label

feature_frame["needs_refresh"] = (
    feature_frame["total_clicks"] < 5
).astype(int)

In [ ]:
# Honest Model

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = feature_frame[
    [
        "total_impressions",
        "avg_position",
        "total_sessions",
        "total_scroll_events"
    ]
]

y = feature_frame["needs_refresh"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(random_state=42)

model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Honest Accuracy:", accuracy_score(y_test, pred))

Honest Accuracy: 0.9280864546791898


In [ ]:
# Leaking Model

X_leak = feature_frame[
    [
        "total_impressions",
        "avg_position",
        "total_sessions",
        "total_scroll_events",
        "needs_refresh"
    ]
]

X_train, X_test, y_train, y_test = train_test_split(
    X_leak,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

leak_model = RandomForestClassifier(random_state=42)

leak_model.fit(X_train, y_train)

leak_pred = leak_model.predict(X_test)

print("Leaking Accuracy:", accuracy_score(y_test, leak_pred))

Leaking Accuracy: 1.0


## Leakage Demonstration

The first model used only historical performance features.

The second model intentionally included the target label as an input feature.

Including the label caused unrealistically high performance because the model had access to the answer it was trying to predict.

This demonstrates why label-derived information must never be used as a feature in a real machine learning workflow.

## Limitation

This notebook analyzes only March 2026.

Using a single month does not capture seasonal trends or longer-term changes in content performance.

## Self-Check

- ✅ Five data contract questions answered.
- ✅ Three verification queries completed.
- ✅ Data grain verified.
- ✅ Availability checked using `IS TRUE`.
- ✅ Five-feature frame created.
- ✅ Each feature justified as available at the decision moment.
- ✅ Leakage demonstrated and explained.
- ✅ One limitation documented.